Retrieval Augmented Generation (RAG) Bot

In [5]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

"""""
img = mpimg.imread('process.png')
plt.imshow(img)
plt.axis('off')
"""

'""\nimg = mpimg.imread(\'process.png\')\nplt.imshow(img)\nplt.axis(\'off\')\n'

Content Extraction 

In [3]:

import pdfplumber
import os

PDF_FOLDER = os.path.join(os.getcwd(), 'raw_documents')
OUTPUT_FOLDER = os.path.join(os.getcwd(), "extracted_texts")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def extract_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += (page.extract_text() or "") + "\n"
    return text

for filename in os.listdir(PDF_FOLDER):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(PDF_FOLDER, filename)
        print("Extracting:", pdf_path)

        text = extract_pdf(pdf_path)

        # Save text file for this PDF
        output_path = os.path.join(OUTPUT_FOLDER, filename.replace(".pdf", ".txt"))
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)



Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 1 Part 1.pdf
Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 1 part 2.pdf
Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 1 part 3.pdf
Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 10 PART 1.pdf
Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 10 PART 2.pdf
Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 10 PART 3.pdf
Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 11 PART 1.pdf
Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 11 PART 2.pdf
Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 11 PART 3.pdf
Extracting: c:\Users\Migs\Desktop\ragbot

Chunking

In [1]:
import os
import json
# NEW: Specialized import paths
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import TokenTextSplitter

# ============================================================
#                   HYPERPARAMETER CONFIG
# ============================================================
CONFIG = {
    # Using the updated HuggingFace model pathing
    "model_name": "sentence-transformers/all-MiniLM-L6-v2",

    # Semantic Chunking Parameters
    "semantic_min_chunk_size":        300,
    "semantic_breakpoint_type":       "percentile",
    "semantic_breakpoint_amount":     80,

    # Token-based Chunking Parameters
    "token_chunk_size":   400,
    "token_overlap":       100,
    "tokenizer_name":     "cl100k_base",

    # Paths
    "text_folder":   os.path.join(os.getcwd(), "extracted_texts"),
    "chunks_folder": os.path.join(os.getcwd(), "chunks_json"),
}
os.makedirs(CONFIG["chunks_folder"], exist_ok=True)

# -------------------- LOAD EMBEDDINGS --------------------
# Updated to use langchain_huggingface class
embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG["model_name"]
)

# -------------------- SEMANTIC CHUNKER --------------------
semantic_chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type=CONFIG["semantic_breakpoint_type"],
    breakpoint_threshold_amount=CONFIG["semantic_breakpoint_amount"],
    # min_chunk_size is supported in recent experimental versions
)

# -------------------- TOKEN SPLITTER --------------------
# Updated to use langchain_text_splitters class
token_splitter = TokenTextSplitter(
    chunk_size=CONFIG["token_chunk_size"],
    chunk_overlap=CONFIG["token_overlap"],
    encoding_name=CONFIG["tokenizer_name"]
)

# ============================================================
#                   PROCESSING LOOP
# ============================================================
for filename in os.listdir(CONFIG["text_folder"]):
    if filename.endswith(".txt"):
        file_path = os.path.join(CONFIG["text_folder"], filename)
        print(f"Chunking: {file_path}")

        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        # Stage 1: Semantic chunking (Now returns List[str] or List[Document])
        semantic_chunks = semantic_chunker.split_text(text)

        # Stage 2: Token chunking for safety/uniformity
        final_chunks = []
        for sc in semantic_chunks:
            final_chunks.extend(token_splitter.split_text(sc))

        # Save JSON output
        base_name = filename.replace(".txt", "")
        json_path = os.path.join(CONFIG["chunks_folder"], f"{base_name}_chunks.json")

        json_output = [
            {
                "doc_id": base_name,
                "chunk_id": f"{base_name}_chunk_{i}",
                "chunk_index": i,
                "text": chunk
            }
            for i, chunk in enumerate(final_chunks)
        ]

        with open(json_path, "w", encoding="utf-8") as out:
            json.dump(json_output, out, indent=4, ensure_ascii=False)

        print(f"Saved {len(final_chunks)} chunks → {json_path}")

c:\Users\Migs\Desktop\ragbot\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 1 Part 1.txt
Saved 28 chunks → c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\chunks_json\Topic 1 Part 1_chunks.json
Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 1 part 2.txt
Saved 28 chunks → c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\chunks_json\Topic 1 part 2_chunks.json
Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 1 part 3.txt
Saved 35 chunks → c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\chunks_json\Topic 1 part 3_chunks.json
Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 10 PART 1.txt
Saved 128 chunks → c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\chunks_json\Topic 10 PART 1_chunks.json
Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 10 PART 

In [4]:
import os
import json
from langchain_together import TogetherEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import TokenTextSplitter

# ============================================================
#                  HYPERPARAMETER CONFIG
# ============================================================
CONFIG = {
    # THE ALIBABA CHOICE: Fastest and most modern for 2026
    "model_name": "Alibaba-NLP/gte-modernbert-base", 
    
    # Together API Key (Required)
    "together_api_key": "f093074f102974466d625db36d8bd171b92df916fa78eb7b91faa9108e6ed5c2",

    # Semantic Chunking Parameters
    "semantic_min_chunk_size":        300,
    "semantic_breakpoint_type":       "percentile",
    # GTE is very sharp at boundaries; 80 is a strong starting point
    "semantic_breakpoint_amount":     80, 

    # Token-based Chunking Parameters
    "token_chunk_size":   400, # ModernBERT handles 8192, so we can use larger safe-buffers
    "token_overlap":       100,
    "tokenizer_name":     "cl100k_base",

    # Paths
    "text_folder":   os.path.join(os.getcwd(), "extracted_texts"),
    "chunks_folder": os.path.join(os.getcwd(), "chunks_json"),
}

os.makedirs(CONFIG["chunks_folder"], exist_ok=True)

# -------------------- LOAD EMBEDDINGS (Together AI) --------------------
# This connects to Together's GTE-ModernBERT endpoint
embeddings = TogetherEmbeddings(
    model=CONFIG["model_name"],
    together_api_key=CONFIG["together_api_key"]
)

# -------------------- SEMANTIC CHUNKER --------------------
semantic_chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type=CONFIG["semantic_breakpoint_type"],
    breakpoint_threshold_amount=CONFIG["semantic_breakpoint_amount"]
)

# -------------------- TOKEN SPLITTER --------------------
token_splitter = TokenTextSplitter(
    chunk_size=CONFIG["token_chunk_size"],
    chunk_overlap=CONFIG["token_overlap"],
    encoding_name=CONFIG["tokenizer_name"]
)

# ============================================================
#                   PROCESSING LOOP
# ============================================================
# ... (your loop continues as before)
# ============================================================
#                   PROCESSING LOOP
# ============================================================
for filename in os.listdir(CONFIG["text_folder"]):
    if filename.endswith(".txt"):
        file_path = os.path.join(CONFIG["text_folder"], filename)
        print(f"Chunking: {file_path}")

        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        # Stage 1: Semantic chunking (Now returns List[str] or List[Document])
        semantic_chunks = semantic_chunker.split_text(text)

        # Stage 2: Token chunking for safety/uniformity
        final_chunks = []
        for sc in semantic_chunks:
            final_chunks.extend(token_splitter.split_text(sc))

        # Save JSON output
        base_name = filename.replace(".txt", "")
        json_path = os.path.join(CONFIG["chunks_folder"], f"{base_name}_chunks.json")

        json_output = [
            {
                "doc_id": base_name,
                "chunk_id": f"{base_name}_chunk_{i}",
                "chunk_index": i,
                "text": chunk
            }
            for i, chunk in enumerate(final_chunks)
        ]

        with open(json_path, "w", encoding="utf-8") as out:
            json.dump(json_output, out, indent=4, ensure_ascii=False)

        print(f"Saved {len(final_chunks)} chunks → {json_path}")



Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 1 Part 1.txt
Saved 27 chunks → c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\chunks_json\Topic 1 Part 1_chunks.json
Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 1 part 2.txt
Saved 28 chunks → c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\chunks_json\Topic 1 part 2_chunks.json
Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 1 part 3.txt
Saved 33 chunks → c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\chunks_json\Topic 1 part 3_chunks.json
Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 10 PART 1.txt
Saved 127 chunks → c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\chunks_json\Topic 10 PART 1_chunks.json
Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 10 PART 

Hosted Inference Eval Dataset Creation

In [10]:
import json
import os
import random
from together import Together

# ----------------------------
# CONFIG
# ----------------------------
TOGETHER_API_KEY = "f093074f102974466d625db36d8bd171b92df916fa78eb7b91faa9108e6ed5c2"
CHUNKS_FOLDER = "chunks_json"
OUTPUT_FILE = "rag_eval.json"
MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
MAX_TOKENS = 512

# ----------------------------
# INITIALIZE CLIENT
# ----------------------------
client = Together(api_key=TOGETHER_API_KEY)

# ----------------------------
# HELPER FUNCTION
# ----------------------------
def extract_json(text):
    import re
    try:
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception as e:
        print("❌ JSON parsing error:", e)
    return []

# ----------------------------
# ANSWER TYPE DEFINITIONS
# ----------------------------
ANSWER_TYPES = {
    "short_fact": "Answer in ONE concise factual sentence.",
    "descriptive": "Answer in 2–3 sentences describing characteristics or details.",
    "explanatory": "Answer by explaining causes, significance, or context using multiple details."
}

# ----------------------------
# PROCESS CHUNKS
# ----------------------------
all_samples = []

for file_name in os.listdir(CHUNKS_FOLDER):
    if not file_name.endswith(".json"):
        continue

    path = os.path.join(CHUNKS_FOLDER, file_name)
    with open(path, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    # --- RANDOM SAMPLING PER FILE ---
    subset_size = 6
    if len(chunks) > subset_size:
        chunks = random.sample(chunks, subset_size)

    for idx, chunk in enumerate(chunks):
        if isinstance(chunk, str):
            chunk = {"text": chunk}

        evidence_text = chunk["text"]

        # Randomly choose answer type
        answer_type = random.choice(list(ANSWER_TYPES.keys()))
        answer_instruction = ANSWER_TYPES[answer_type]

        # Prompt
        prompt = f"""
You are given a historical passage.

Generate EXACTLY ONE question that:
- Can be answered ONLY using the information in the passage
- Does NOT rely on external knowledge
- Does NOT refer to the passage, text, or document explicitly
- Is NOT a yes/no question

{answer_instruction}

Return STRICT JSON in the following format:

[
  {{
    "query": "...",
    "answer": "..."
  }}
]

Text:
\"\"\"
{evidence_text}
\"\"\"
"""

        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": "You are an expert historian who generates high-quality question–answer pairs from historical texts."
                    },
                    {"role": "user", "content": prompt}
                ],
                temperature=0.2,
                max_tokens=MAX_TOKENS
            )

            output_text = response.choices[0].message.content
            data = extract_json(output_text)

            if data:
                item = data[0]
                item["id"] = f"{file_name}_sample_{idx}"
                item["answer_type"] = answer_type
                item["evidence_passages"] = [evidence_text]
                all_samples.append(item)

        except Exception as e:
            print(f"❌ Error processing chunk: {e}")

# ----------------------------
# SAVE RESULTS
# ----------------------------
with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    json.dump(all_samples, out, indent=4, ensure_ascii=False)

print(f"\n✅ RAG evaluation dataset saved to {OUTPUT_FILE}")
print(f"Total samples: {len(all_samples)}")


╭───────────────────────────────────────────── 🚀 New SDK Available ──────────────────────────────────────────────╮
│ Together Python SDK 2.0 is now available!                                                                       │
│                                                                                                                 │
│ Install the beta:                                                                                               │
│ pip install --pre together  or  uv add together --prerelease allow                                              │
│                                                                                                                 │
│ New SDK: ]8;id=431359;https://github.com/togethercomputer/together-py\https://github.com/togethercomputer/together-py]8;;\                                                        │
│ Migration guide: ]8;id=174107;https://docs.together.ai/docs/pythonv2-migration-guide\https://docs.together.ai/docs/pythonv2-migration-guide]8;;\                                         │
│                                                                                                                 │
│ This package will be maintained until January 2026.                                                             │
│ Set TOGETHER_NO_BANNER=1 to hide this message.                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ RAG evaluation dataset saved to rag_eval.json
Total samples: 198


Embedding

In [3]:
import os
import json
# NEW: Partner package for HuggingFace
from langchain_huggingface import HuggingFaceEmbeddings
# NEW: Community package for FAISS
from langchain_community.vectorstores import FAISS
# NEW: Standard Document class
from langchain_core.documents import Document


# --------- PATHS ----------
CHUNKS_FOLDER = "chunks_json"
FAISS_INDEX_PATH = "faiss_index"

# --------- EMBEDDINGS ----------
# langchain_huggingface is now the standard over langchain.embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding model loaded.")

# --------- LOAD CHUNK DATA ----------
documents = []

for filename in os.listdir(CHUNKS_FOLDER):
    if filename.endswith("_chunks.json"):
        file_path = os.path.join(CHUNKS_FOLDER, filename)
        print(f"📄 Loading: {file_path}")

        with open(file_path, "r", encoding="utf-8") as f:
            chunks = json.load(f)

        for chunk in chunks:
            # NEW: Instead of a dict, we create a proper Document object
            doc = Document(
                page_content=chunk["text"],
                metadata={
                    "doc_id": chunk["doc_id"],
                    "chunk_id": chunk["chunk_id"],
                    "chunk_index": chunk["chunk_index"]
                }
            )
            documents.append(doc)

print(f"✅ Loaded {len(documents)} chunks")

# --------- CREATE & SAVE FAISS INDEX ----------
if documents:
    print("⏳ Creating FAISS index (this may take a minute)...")
    vector_store = FAISS.from_documents(documents, embedding_model)
    
    # Save the index locally so you don't have to re-embed next time
    vector_store.save_local(FAISS_INDEX_PATH)
    print(f"✅ FAISS index saved to '{FAISS_INDEX_PATH}'")

✅ Embedding model loaded.
📄 Loading: chunks_json\Topic 1 Part 1_chunks.json
📄 Loading: chunks_json\Topic 1 part 2_chunks.json
📄 Loading: chunks_json\Topic 1 part 3_chunks.json
📄 Loading: chunks_json\Topic 10 PART 1_chunks.json
📄 Loading: chunks_json\Topic 10 PART 2_chunks.json
📄 Loading: chunks_json\Topic 10 PART 3_chunks.json
📄 Loading: chunks_json\Topic 11 PART 1_chunks.json
📄 Loading: chunks_json\Topic 11 PART 2_chunks.json
📄 Loading: chunks_json\Topic 11 PART 3_chunks.json
📄 Loading: chunks_json\Topic 2 part 1_chunks.json
📄 Loading: chunks_json\Topic 2 Part 2_chunks.json
📄 Loading: chunks_json\Topic 2 Part 3_chunks.json
📄 Loading: chunks_json\Topic 3 Part 1_chunks.json
📄 Loading: chunks_json\Topic 3 Part 2_chunks.json
📄 Loading: chunks_json\Topic 3 Part 3_chunks.json
📄 Loading: chunks_json\Topic 4 Part 1_chunks.json
📄 Loading: chunks_json\Topic 4 Part 2_chunks.json
📄 Loading: chunks_json\Topic 4 Part 3_chunks.json
📄 Loading: chunks_json\Topic 5 Part 1_chunks.json
📄 Loading: chunks_

In [5]:
import os
import json
# NEW: Import for Together AI integration
from langchain_together import TogetherEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# --------- PATHS ----------
CHUNKS_FOLDER = "chunks_json"
FAISS_INDEX_PATH = "faiss_index"

# --------- EMBEDDINGS (Together AI + Alibaba) ----------
# Replace HuggingFace with Together AI's endpoint
embedding_model = TogetherEmbeddings(
    model="Alibaba-NLP/gte-modernbert-base",
    # Ensure TOGETHER_API_KEY is in your environment variables
    together_api_key= "f093074f102974466d625db36d8bd171b92df916fa78eb7b91faa9108e6ed5c2" 
)
print("✅ Together AI Embedding model (GTE-ModernBERT) loaded.")

# --------- LOAD CHUNK DATA ----------
documents = []
for filename in os.listdir(CHUNKS_FOLDER):
    if filename.endswith("_chunks.json"):
        file_path = os.path.join(CHUNKS_FOLDER, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            chunks = json.load(f)

        for chunk in chunks:
            doc = Document(
                page_content=chunk["text"],
                metadata={
                    "doc_id": chunk["doc_id"],
                    "chunk_id": chunk["chunk_id"],
                    "chunk_index": chunk["chunk_index"]
                }
            )
            documents.append(doc)

print(f"✅ Loaded {len(documents)} chunks")

# --------- CREATE & SAVE FAISS INDEX ----------
if documents:
    print("⏳ Sending chunks to Together AI for embedding (Cloud-based)...")
    # This now sends batches to the API instead of using your local CPU/GPU
    vector_store = FAISS.from_documents(documents, embedding_model)
    
    vector_store.save_local(FAISS_INDEX_PATH)
    print(f"✅ FAISS index saved to '{FAISS_INDEX_PATH}'")

✅ Together AI Embedding model (GTE-ModernBERT) loaded.
✅ Loaded 2049 chunks
⏳ Sending chunks to Together AI for embedding (Cloud-based)...
✅ FAISS index saved to 'faiss_index'


Retrieval

In [12]:
import json
import os
# NEW: Import from the dedicated huggingface and community packages
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ---- Load Embeddings (MUST MATCH INDEX) ----
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding model loaded.")

# ---- Load FAISS Vector Store ----
FAISS_FOLDER = "faiss_index"

# Ensure the path exists before loading
if not os.path.exists(FAISS_FOLDER):
    raise FileNotFoundError(f"The folder {FAISS_FOLDER} does not exist.")

vectorstore = FAISS.load_local(
    FAISS_FOLDER,
    embeddings,
    allow_dangerous_deserialization=True  # Required for security in modern versions
)

# ---- Create Retriever ----
# k: number of final docs, fetch_k: number of docs to pass to MMR algorithm
retriever = vectorstore.as_retriever(

    search_kwargs={
        "k": 1,        # Final results returned
 
    }
)

# ---- Load Evaluation Queries ----
with open("rag_eval.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# ---- Run Retrieval for Each Query ----
retrieval_results = []

for item in eval_data:
    query = item["query"]

    # MODERN CHANGE: Use .invoke() instead of .get_relevant_documents()
    docs = retriever.invoke(query)

    retrieval_results.append({
        "id": item.get("id"),
        "query": query,
        "retrieved_chunks": [
            {
                "content": doc.page_content,
                "metadata": doc.metadata
            }
            for doc in docs
        ]
    })

print(f"✅ Retrieved documents for {len(retrieval_results)} queries.")

# ---- Save Retrieval Output ----
OUTPUT_FILE = "retrieval_output.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(retrieval_results, f, indent=2, ensure_ascii=False)

print(f"✅ Retrieval results saved to {OUTPUT_FILE}")

✅ Embedding model loaded.
✅ Retrieved documents for 198 queries.
✅ Retrieval results saved to retrieval_output.json


In [19]:
import json
import os
# NEW: Import Together AI integration
from langchain_together import TogetherEmbeddings
from langchain_community.vectorstores import FAISS

# ---- Load Embeddings (Together AI + Alibaba) ----
# This MUST match the model used to create the index
embeddings = TogetherEmbeddings(
    model="Alibaba-NLP/gte-modernbert-base",
    together_api_key= "f093074f102974466d625db36d8bd171b92df916fa78eb7b91faa9108e6ed5c2"
)
print("✅ Together AI Embedding model (GTE-ModernBERT) loaded.")

# ---- Load FAISS Vector Store ----
FAISS_FOLDER = "faiss_index"

if not os.path.exists(FAISS_FOLDER):
    raise FileNotFoundError(f"The folder {FAISS_FOLDER} does not exist. Ensure you have run the indexing script first.")

# Loading the index that was built with Alibaba GTE
vectorstore = FAISS.load_local(
    FAISS_FOLDER,
    embeddings,
    allow_dangerous_deserialization=True 
)

# ---- Create Retriever ----
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3, # Retrieve top 10 most relevant chunks
    }
)

# ---- Load Evaluation Queries ----
# Ensure rag_eval.json contains your questions on Philippine cultural history
with open("rag_eval.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# ---- Run Retrieval for Each Query ----
retrieval_results = []

for item in eval_data:
    query = item["query"]
    print(f"🔍 Retrieving for: {query[:50]}...")

    # .invoke() sends the query to Together AI to get the query vector
    docs = retriever.invoke(query)

    retrieval_results.append({
        "id": item.get("id"),
        "query": query,
        "retrieved_chunks": [
            {
                "content": doc.page_content,
                "metadata": doc.metadata
            }
            for doc in docs
        ]
    })

print(f"✅ Retrieved documents for {len(retrieval_results)} queries.")

# ---- Save Retrieval Output ----
OUTPUT_FILE = "retrieval_output.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(retrieval_results, f, indent=2, ensure_ascii=False)

print(f"✅ Retrieval results saved to {OUTPUT_FILE}")

✅ Together AI Embedding model (GTE-ModernBERT) loaded.
🔍 Retrieving for: What species of Homo was established based on the ...
🔍 Retrieving for: Where is Easter Island located...
🔍 Retrieving for: What are the dates of the early modern human remai...
🔍 Retrieving for: What degree does Hernandez hold...
🔍 Retrieving for: What geographical locations did the Austronesian l...
🔍 Retrieving for: What is the title of the historical work being ref...
🔍 Retrieving for: What factors contributed to the Austronesian expan...
🔍 Retrieving for: What was considered a sine qua non for the expansi...
🔍 Retrieving for: What types of containers were used for interring s...
🔍 Retrieving for: What items are offered as presents by a vessel's c...
🔍 Retrieving for: What significant transformations did prehistoric A...
🔍 Retrieving for: What year is mentioned as a point in time when the...
🔍 Retrieving for: What do foreign traders receive in order to serve ...
🔍 Retrieving for: What products are exported fro

Retrieval Evaluation

In [13]:
import json

# ==========================================
# CONFIGURATION
# ==========================================
RETRIEVAL_FILE = "retrieval_output.json"   # The file you just generated
EVAL_FILE = "rag_eval.json" # Your gold standard dataset (check filename!)
OUTPUT_LOG = "evaluation_metrics.json"

# ==========================================
# 1. HELPER: TEXT NORMALIZATION
# ==========================================
def normalize_text(text):
    """
    Standardizes text by removing newlines, extra spaces, and lowercasing.
    Example: "Hello   World\n" -> "hello world"
    """
    if not text:
        return ""
    # Replace newlines/tabs with space, then collapse multiple spaces
    text = " ".join(text.split()) 
    return text.lower().strip()

# ==========================================
# 2. LOAD DATA
# ==========================================
print("📂 Loading data files...")

with open(RETRIEVAL_FILE, "r", encoding="utf-8") as f:
    retrieval_data = json.load(f)

with open(EVAL_FILE, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# Convert retrieval list to a dictionary for faster lookup (Key = Query)
# This handles cases where the order might differ
retrieval_map = {item['query']: item['retrieved_chunks'] for item in retrieval_data}

# ==========================================
# 3. RUN EVALUATION
# ==========================================
print("🚀 Running comparison...")

total_hits = 0
total_mrr = 0
total_queries = 0
detailed_logs = []

for eval_item in eval_data:
    query = eval_item['query']
    
    # "evidence_passages" or "evidence_text" depending on your exact json key
    # Based on your snippet, you used "evidence_passages" in one and "evidence_text" in another.
    # We try both to be safe.
    gold_passages = eval_item.get('evidence_passages', eval_item.get('evidence_text', []))
    
    # Skip if no evidence provided (rare)
    if not gold_passages:
        continue

    total_queries += 1

    # Get the retrieved docs for this specific query
    retrieved_docs = retrieval_map.get(query, [])
    
    is_hit = False
    rank = 0
    matched_passage = ""

    # Pre-normalize all retrieved chunks for this query
    norm_retrieved = [normalize_text(doc['content']) for doc in retrieved_docs]

    # CHECK: Does ANY gold passage exist inside the retrieved list?
    for gold in gold_passages:
        norm_gold = normalize_text(gold)
        
        # We check exact inclusion in the list
        # (Since you are using chunks, exact string match of the chunk text is best)
        if norm_gold in norm_retrieved:
            print(norm_gold)
            print(norm_retrieved)
            is_hit = True
            rank = norm_retrieved.index(norm_gold) + 1 # 1-based rank
            matched_passage = gold + "..." # Log snippet
            break
    
    # Update Metrics
    if is_hit:
        total_hits += 1
        total_mrr += (1.0 / rank)
        status = "HIT"
    else:
        status = "MISS"
        rank = 0

    # Log details
    detailed_logs.append({
        "query": query,
        "status": status,
        "rank": rank,
        "matched_passage": matched_passage
    })

# ==========================================
# 4. FINAL REPORT
# ==========================================
if total_queries > 0:
    hit_rate = total_hits / total_queries
    mrr_score = total_mrr / total_queries
else:
    hit_rate = 0
    mrr_score = 0

print("\n" + "="*40)
print("📊 RETRIEVAL METRICS REPORT")
print("="*40)
print(f"Total Queries Evaluated: {total_queries}")
print(f"Hit Rate (Recall):       {hit_rate:.2%}")
print(f"MRR Score:               {mrr_score:.4f}")
print("="*40)
print(f"✅ Detailed logs saved to {OUTPUT_LOG}")

with open(OUTPUT_LOG, "w", encoding="utf-8") as f:
    json.dump(detailed_logs, f, indent=4, ensure_ascii=False)

📂 Loading data files...
🚀 Running comparison...
hernandez, ph.d.
['hernandez, ph.d.']
moving beyond austro-tai into austronesian proper, the reconstruction of linguistic prehistory which is most widely used today is that postulated by robert blust (1884-5). this is based on a “family-tree” of subgroups and a hierarchy of proto-languages extending from proto austronesian (pan) forwards in time. reduced to its essentials, blust’s reconstruction favors a geographical expansion beginning in taiwan (the location of the oldest austronesian languages including pan), then encompassing the philippines, borneo and sulawesi, and finally bifurcating, one branch moving west to java, sumatra and malaya, the other moving east into oceania (see darrell tyron’s more detailed summary in this volume). a wealth of linguistic detail can be added to this framework but here i will restrict myself to some implications of broad historical and cultural significance. during the linguistic stage before the break 

Generation

In [17]:
import json
import random
from together import Together

# ----------------------------
# CONFIG
# ----------------------------
TOGETHER_API_KEY = "f093074f102974466d625db36d8bd171b92df916fa78eb7b91faa9108e6ed5c2"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"  # hosted model
MAX_TOKENS = 100

# ----------------------------
# INITIALIZE CLIENT
# ----------------------------
client = Together(api_key=TOGETHER_API_KEY)

# ---- Generation Results Container ----
generation_results = []

for item in retrieval_results:
    query = item["query"]

    # ---- Combine Retrieved Context ----
    context = "\n\n".join(
        [chunk["content"] for chunk in item["retrieved_chunks"]]
    )

    # ---- Prompt Template ----
    prompt = f"""

Answer the question using ONLY the provided context.
If the answer is not present, say:
"The information is not available in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    try:
        # ---- Generate Answer using Together Hosted Model ----
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are an expert academic assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,
            max_tokens=MAX_TOKENS
        )

        generated_text = response.choices[0].message.content.strip()

        generation_results.append({
            "id": item.get("id"),
            "query": query,
            "answer": generated_text,
            "used_chunks": item["retrieved_chunks"]
        })

    except Exception as e:
        print(f"❌ Error generating answer for query '{query}': {e}")

# ---- Save Results ----
with open("generation_output.json", "w", encoding="utf-8") as f:
    json.dump(generation_results, f, indent=2, ensure_ascii=False)

print(f"✅ Generated answers for {len(generation_results)} queries.")


✅ Generated answers for 198 queries.


Judge

In [16]:
import json
import re
from together import Together

# ---- Load Files ----
with open("rag_eval.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

with open("generation_output.json", "r", encoding="utf-8") as f:
    gen_data = json.load(f)

# ---- Map eval answers by query ----
eval_map = {item["query"]: item for item in eval_data}

# ---- Initialize Together AI Client ----
TOGETHER_API_KEY = "f093074f102974466d625db36d8bd171b92df916fa78eb7b91faa9108e6ed5c2"
client = Together(api_key=TOGETHER_API_KEY)
MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct-Turbo"  # or another Together hosted model

# ---- Prompt Template ----
def judge_prompt(question, reference, generated, context):
    return f"""
### Role
You are an expert auditor specializing in Retrieval-Augmented Generation (RAG) systems. Your task is to provide a rigorous, objective evaluation of a GENERATED ANSWER.

### Inputs
**Question:** {question}

**Retrieved Context (The only source of truth):** {context}

**Reference Answer (The ideal response):** {reference}

**Generated Answer (To be evaluated):** {generated}

### Evaluation Criteria & Rubric

#### 1. Faithfulness (Groundedness)
- **Definition:** Is every claim in the answer supported by the Context?
- **Score 1:** Major hallucinations; contains information that contradicts the context.
- **Score 3:** Partially grounded; contains some claims not found in the context.
- **Score 5:** Perfectly grounded; every single sentence is supported by the provided context.

#### 2. Correctness
- **Definition:** How well does the generated answer match the factual content of the Reference Answer?
- **Score 1:** Factually wrong or completely irrelevant.
- **Score 3:** Captures the main idea but misses key nuances.
- **Score 5:** Fully accurate and matches the reference answer's meaning.

#### 3. Completeness
- **Definition:** Does the answer address all parts of the user's question?
- **Score 1:** Extremely brief or ignores most of the question.
- **Score 5:** Thorough; provides a comprehensive response to the query.

### Response Format
You must think step-by-step. First, analyze the alignment between the inputs. Then, provide the scores in the following JSON format:

{{
  "thought_process": "<your step-by-step reasoning for each metric>",
  "faithfulness": <int>,
  "correctness": <int>,
  "completeness": <int>,
  "verdict": "<final brief summary>"
}}
"""

# ---- Helper function ----
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(match.group()) if match else None

# ---- Evaluation Loop ----
judge_results = []

for item in gen_data:
    query = item["query"]
    generated = item["answer"]

    eval_item = eval_map.get(query)
    if not eval_item:
        continue

    reference = eval_item["answer"]

    context = "\n\n".join(
        chunk["content"] for chunk in item["used_chunks"]
    )

    prompt = judge_prompt(query, reference, generated, context)

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are an impartial academic evaluator who scores answers objectively."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=512
        )

        verdict_text = response.choices[0].message.content
        scores = extract_json(verdict_text)

        judge_results.append({
            "query": query,
            "reference_answer": reference,
            "generated_answer": generated,
            "judge_scores": scores
        })

    except Exception as e:
        print(f"❌ Error judging query '{query}': {e}")

# ---- Save Results ----
with open("llm_judge_results.json", "w", encoding="utf-8") as f:
    json.dump(judge_results, f, indent=2, ensure_ascii=False)

print(f"✅ Judged {len(judge_results)} answers using Together AI.")


✅ Judged 0 answers using Together AI.


In [22]:
import json
import re
from together import Together

# ---- Configuration ----
SUBSET_SIZE = 5  # Set to None or 0 to run all
TOGETHER_API_KEY = "f093074f102974466d625db36d8bd171b92df916fa78eb7b91faa9108e6ed5c2"
MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

# ---- Load Files ----
print("📂 Loading data files...")
with open("rag_eval.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

with open("generation_output.json", "r", encoding="utf-8") as f:
    gen_data = json.load(f)

# ---- Map eval answers by query ----
eval_map = {item["query"]: item for item in eval_data}

# ---- Initialize Together AI Client ----
client = Together(api_key=TOGETHER_API_KEY)

# ---- Prompt Template ----
def judge_prompt(question, reference, generated, context):
    return f"""
### Role
You are an expert auditor specializing in Retrieval-Augmented Generation (RAG) systems. Your task is to provide a rigorous, objective evaluation of a GENERATED ANSWER.

### Inputs
**Question:** {question}

**Retrieved Context (The only source of truth):** {context}

**Reference Answer (The ideal response):** {reference}

**Generated Answer (To be evaluated):** {generated}

### Evaluation Criteria & Rubric

#### 1. Faithfulness (Groundedness)
- **Definition:** Is every claim in the answer supported by the Context?
- **Score 1:** Major hallucinations; contains information that contradicts the context.
- **Score 3:** Partially grounded; contains some claims not found in the context.
- **Score 5:** Perfectly grounded; every single sentence is supported by the provided context.

#### 2. Correctness
- **Definition:** How well does the generated answer match the factual content of the Reference Answer?
- **Score 1:** Factually wrong or completely irrelevant.
- **Score 3:** Captures the main idea but misses key nuances.
- **Score 5:** Fully accurate and matches the reference answer's meaning.

#### 3. Completeness
- **Definition:** Does the answer address all parts of the user's question?
- **Score 1:** Extremely brief or ignores most of the question.
- **Score 5:** Thorough; provides a comprehensive response to the query.

### Response Format
You must think step-by-step. First, analyze the alignment between the inputs. Then, provide the scores in the following JSON format. Do not add conversational text outside the JSON.

{{
  "thought_process": "<your step-by-step reasoning for each metric>",
  "faithfulness": <int>,
  "correctness": <int>,
  "completeness": <int>,
  "verdict": "<final brief summary>"
}}
"""

# ---- Robust JSON Extractor ----
def extract_json(text):
    """
    Attempts to extract and parse JSON from a string, handling common LLM errors.
    """
    if not text:
        return None

    # 1. Try to find JSON inside Markdown code blocks (```json ... ```)
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        json_str = match.group(1)
    else:
        # 2. If no code block, try to find the outer braces manually
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            json_str = match.group(0)
        else:
            print(f"⚠️  No JSON braces found in response: {text[:100]}...")
            return None

    # 3. Clean up common LLM syntax errors
    # Remove trailing commas (e.g., {"a": 1,} -> {"a": 1})
    json_str = re.sub(r",\s*\}", "}", json_str) 
    json_str = re.sub(r",\s*\]", "]", json_str)
    
    # Attempt parsing
    try:
        # strict=False allows control characters like newlines inside strings
        return json.loads(json_str, strict=False)
    except json.JSONDecodeError:
        # 4. Fallback: Sometimes LLMs use single quotes (') instead of double (")
        # We try replacing them, but we must be careful not to break internal apostrophes
        try:
            # Simple heuristic: if it looks like python dict str, try swapping quotes
            fixed_str = json_str.replace("'", '"')
            return json.loads(fixed_str, strict=False)
        except json.JSONDecodeError as e:
            print(f"\n❌ JSON Parse Failed!")
            print(f"   Error: {e}")
            print(f"   Bad String snippet: {json_str[:200]}...\n")
            return None

# ---- Evaluation Loop ----
judge_results = []

# Determine the dataset to process based on SUBSET_SIZE
data_to_process = gen_data[:SUBSET_SIZE] if SUBSET_SIZE else gen_data

print(f"🚀 Starting evaluation on {len(data_to_process)} items...")

for i, item in enumerate(data_to_process):
    query = item["query"]
    generated = item["answer"]

    eval_item = eval_map.get(query)
    if not eval_item:
        continue

    reference = eval_item["answer"]

    # Combine chunks into one context string
    context = "\n\n".join(
        chunk["content"] for chunk in item.get("used_chunks", [])
    )

    prompt = judge_prompt(query, reference, generated, context)

    print(f"[{i+1}/{len(data_to_process)}] Judging query: {query[:50]}...")

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are an impartial academic evaluator who scores answers objectively. Output ONLY valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=1024 # Increased token limit for longer reasoning
        )

        verdict_text = response.choices[0].message.content
        scores = extract_json(verdict_text)

        if scores is None:
            print(f"   ⚠️  Warning: Received NULL scores for query '{query[:30]}...'")
            # Create a placeholder error object so the pipeline doesn't break
            scores = {
                "thought_process": "JSON Parsing Failed", 
                "faithfulness": 0, 
                "correctness": 0, 
                "completeness": 0, 
                "verdict": "Error"
            }

        judge_results.append({
            "query": query,
            "reference_answer": reference,
            "generated_answer": generated,
            "judge_scores": scores
        })

    except Exception as e:
        print(f"   ❌ API Error judging query '{query}': {e}")

# ---- Save Results ----
output_filename = "llm_judge_results.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(judge_results, f, indent=2, ensure_ascii=False)

print(f"\n✅ Judged {len(judge_results)} answers.")
print(f"✅ Results saved to {output_filename}")

📂 Loading data files...
🚀 Starting evaluation on 5 items...
[1/5] Judging query: What species of Homo was established based on the ...
[2/5] Judging query: Where is Easter Island located...
[3/5] Judging query: What are the dates of the early modern human remai...
[4/5] Judging query: What degree does Hernandez hold...
[5/5] Judging query: What geographical locations did the Austronesian l...

✅ Judged 5 answers.
✅ Results saved to llm_judge_results.json


Automatic Metrics

In [19]:
import json
import numpy as np
import evaluate

# ==========================================
# 1. LOAD DATA
# ==========================================
# This should be your generation_output.json or the results from your previous script
with open("llm_judge_results.json", "r", encoding="utf-8") as f:
    results_data = json.load(f)

# Extract predictions and references
predictions = [item["generated_answer"] for item in results_data]
references = [item["reference_answer"] for item in results_data]

print(f"📊 Running metrics for {len(predictions)} samples...")

# ==========================================
# 2. INITIALIZE METRICS
# ==========================================
# 'evaluate' is the modern HuggingFace library for metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# ==========================================
# 3. CALCULATE
# ==========================================

# BLEU Score
bleu_results = bleu.compute(predictions=predictions, references=[[r] for r in references])

# ROUGE Score (L, 1, 2)
rouge_results = rouge.compute(predictions=predictions, references=references)

# BERTScore (Semantics-based)
# 'lang="en"' is required. It uses a RoBERTa model by default.
bert_results = bertscore.compute(
    predictions=predictions, 
    references=references, 
    lang="en",
    model_type="distilbert-base-uncased" # Faster/lighter than default
)

# ==========================================
# 4. AGGREGATE & SAVE
# ==========================================

# Calculate mean of BERTScore (it returns a list per sample)
avg_bert_f1 = np.mean(bert_results["f1"])

final_report = {
    "summary": {
        "sacrebleu": round(bleu_results["score"], 2),
        "rouge1": round(rouge_results["rouge1"], 4),
        "rouge2": round(rouge_results["rouge2"], 4),
        "rougeL": round(rouge_results["rougeL"], 4),
        "bertscore_f1_avg": round(float(avg_bert_f1), 4)
    },
    "per_sample_bertscore": bert_results["f1"]
}

print("\n" + "="*40)
print("📈 AUTOMATIC METRICS REPORT")
print("="*40)
print(f"SacreBLEU:  {final_report['summary']['sacrebleu']}")
print(f"ROUGE-L:    {final_report['summary']['rougeL']}")
print(f"BERTScore:  {final_report['summary']['bertscore_f1_avg']}")
print("="*40)

with open("auto_metrics_results.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=4)

print("✅ Metrics saved to auto_metrics_results.json")

📊 Running metrics for 2 samples...

📈 AUTOMATIC METRICS REPORT
SacreBLEU:  2.38
ROUGE-L:    0.1452
BERTScore:  0.7089
✅ Metrics saved to auto_metrics_results.json
